In [33]:
import collections
from IPython import display
import matplotlib.pyplot as plt

import numpy as np
import torch
import itertools as itr
from torch import nn
from common.ffn.ffn_relu import ParametricReLUNet
from common.coeff_calc.coeff_calc_ntk import NTKSimulator
from sklearn.metrics import mean_squared_error

from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from util import *
import logging

In [34]:
logging.basicConfig(filename="log_twosteps_minst_draft.log",
                level=logging.WARN,
                format='%(levelname)s: %(asctime)s %(message)s',
                datefmt='%m/%d/%Y %I:%M:%S')

In [35]:
# use it to conver from PIL to torch.Tensor
image_transform = ToTensor()

train_dataset = MNIST(root='./', train=True, download=True, transform=image_transform)
test_dataset = MNIST(root='./', train=False, download=True, transform=image_transform)

In [36]:
BATCH_SIZE = 64

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

In [37]:
class MNISTReLU(ParametricReLUNet):
    def __init__(self, input_dim, output_dim):
        super().__init__(n0=input_dim,nk=0,nl=output_dim,l=0, bias_on=True)

        self.input_fc = nn.Linear(input_dim, 250)
        self.hidden_fc = nn.Linear(250, 100)
        self.output_fc = nn.Linear(100, output_dim)

    def forward_(self, x):
        return self.forward(x)

    def forward(self, x):
        #x = [batch size, height, width]
        #x = [batch size, height * width]
        h_1 = self.PReLU(self.input_fc(x))
        #h_1 = [batch size, 250]
        h_2 = self.PReLU(self.hidden_fc(h_1))
        #h_2 = [batch size, 100]
        y_pred = self.output_fc(h_2)
        #y_pred = [batch size, output dim]
        return y_pred
    
    def init_weights(self, cb=0.0, cw=1.0):
        if self.get_log_level() == "debug":
            print("FeedForwardNet weights initialisation with cb={}, cw={}".format(cb, cw))

        #Weight initialisation as in 2.19, 2.20
        self.cb, self.cw = cb, cw
        self.init_linear_weights(self.input_fc, self.bias_on, cb, cw/self.input_fc.in_features)
        self.init_linear_weights(self.hidden_fc, self.bias_on, cb, cw/self.hidden_fc.in_features)
        self.init_linear_weights(self.output_fc, self.bias_on, cb, cw/self.output_fc.in_features)

    def grad_zero(self):
        self.input_fc.weight.grad.zero_()
        self.input_fc.bias.grad.zero_()
        self.hidden_fc.weight.grad.zero_()
        self.hidden_fc.bias.grad.zero_()
        self.output_fc.weight.grad.zero_()
        self.output_fc.bias.grad.zero_()

In [38]:
INPUT_DIM = 28 * 28
OUTPUT_DIM = 10  # num classes

#lw = 7.5 - for BATCH_SIZE = 32
lb, lw = 1e-2, 7.5 #1e-2,1,1 #1e-2, 1e-2, 1
#Weights distribution variances are set as in (5.67)
slope_plus, slope_minus=1.0, 0.0
cb, cw = 0, 2.0/(slope_plus**2.0 + slope_minus**2.0)

testNet = MNISTReLU(INPUT_DIM, OUTPUT_DIM)
testNet.set_log_level("info")
testNet.set_slopes(slope_plus, slope_minus)
testNet.init_weights(cb, cw)


FeedForwardNet created with n0=784, nk=0, nl=10, l=0, bias_on=True


##### Step forward

In [39]:
def labels_to_logits(labels):
    batch_size = labels.shape[0]
    zz = np.full((OUTPUT_DIM, batch_size), -3.0)
    for batch_num in range(batch_size):
        zz[labels[batch_num], batch_num] = 3

    #logging.debug("Logits for iteration {}: {}".format(iteration_num, np.transpose(yy)))
    return zz

images, labels = next(iter(train_dataloader))
yy = labels_to_logits(labels) #true logits ∈ [-3, 3]
#images = [batch_size, height, width]
batch_size = images.shape[0]
xx = images.view(batch_size, -1)
logits = testNet.forward_(xx)
zz = np.transpose(logits.detach().numpy()) #obtained logits


##### NTK by formulas (8.4)

In [40]:
#lambdas as from (8.5)
lw_input, lw_hidden, lw_output = lw/INPUT_DIM, lw/250, lw/100

print("Calculating derivatives")
input_dweight = np.zeros((OUTPUT_DIM, BATCH_SIZE, 250, INPUT_DIM))
input_dbias = np.zeros((OUTPUT_DIM, BATCH_SIZE, 250))
hidden_dweight = np.zeros((OUTPUT_DIM, BATCH_SIZE, 100, 250))
hidden_dbias = np.zeros((OUTPUT_DIM, BATCH_SIZE, 100))
output_dweight = np.zeros((OUTPUT_DIM, BATCH_SIZE, OUTPUT_DIM, 100))
output_dbias = np.zeros((OUTPUT_DIM, BATCH_SIZE, OUTPUT_DIM))

for kk, alpha in itr.product(range(OUTPUT_DIM), range(BATCH_SIZE)):
    df = torch.autograd.grad(logits[alpha,kk], (testNet.input_fc.weight, testNet.input_fc.bias\
                                    , testNet.hidden_fc.weight, testNet.hidden_fc.bias\
                                    , testNet.output_fc.weight, testNet.output_fc.bias)\
        , retain_graph=True, create_graph=True, allow_unused=True)
    input_dweight[kk, alpha] = df[0].detach().numpy()
    input_dbias[kk, alpha] = df[1].detach().numpy()
    hidden_dweight[kk, alpha] = df[2].detach().numpy()
    hidden_dbias[kk, alpha] = df[3].detach().numpy()
    output_dweight[kk, alpha] = df[4].detach().numpy()
    output_dbias[kk, alpha] = df[5].detach().numpy()

print("Calculating NTK by (8.4)")
HL = np.zeros((OUTPUT_DIM, OUTPUT_DIM, BATCH_SIZE, BATCH_SIZE))
for kk1, kk2, alpha1, alpha2 in itr.product(range(OUTPUT_DIM), range(OUTPUT_DIM)\
                                            , range(BATCH_SIZE), range(BATCH_SIZE)):
    if HL[kk1, kk2, alpha1, alpha2] == 0 or HL[kk2, kk1, alpha2, alpha1] == 0:
        val = lb * np.dot(input_dbias[kk1, alpha1], input_dbias[kk2, alpha2])
        val += lw_input * np.sum(np.multiply(input_dweight[kk1, alpha1], input_dweight[kk2, alpha2]))
        val += lb * np.dot(hidden_dbias[kk1, alpha1], hidden_dbias[kk2, alpha2])
        val += lw_hidden * np.sum(np.multiply(hidden_dweight[kk1, alpha1], hidden_dweight[kk2, alpha2]))
        val += lb * np.dot(output_dbias[kk1, alpha1], output_dbias[kk2, alpha2])
        val += lw_output * np.sum(np.multiply(output_dweight[kk1, alpha1], output_dweight[kk2, alpha2]))
        HL[kk1, kk2, alpha1, alpha2] = HL[kk2, kk1, alpha2, alpha1] = val

print("Calculating average and inverted")
HL_avg = np.zeros((BATCH_SIZE, BATCH_SIZE))
HL_delta = np.copy(HL)

for alpha1, alpha2 in itr.product(range(BATCH_SIZE), range(BATCH_SIZE)):
    H_avg = np.average([HL[num, num, alpha1, alpha2] for num in np.arange(OUTPUT_DIM)])
    HL_avg[alpha1, alpha2] = H_avg
    for num in np.arange(OUTPUT_DIM):
        HL_delta[num, num, alpha1, alpha2] = HL[num, num, alpha1, alpha2] - H_avg

HL_avg_top = np.linalg.inv(HL_avg)


Calculating derivatives


Calculating NTK by (8.4)
Calculating average and inverted


##### Calculating Newton tensor by (∞.71), weights update by (∞.68) and checking accuracy

In [41]:
term0 = np.zeros((OUTPUT_DIM, OUTPUT_DIM, BATCH_SIZE, BATCH_SIZE))
for alpha1, alpha2 in itr.product(range(BATCH_SIZE), range(BATCH_SIZE)):
    for num in range(OUTPUT_DIM):
        term0[num, num, alpha1, alpha2] = HL_avg_top[alpha1, alpha2]

term1a = np.tensordot(np.tensordot(HL_avg_top, HL_delta, axes=((1),(2))), HL_avg_top, axes=((3),(0)))
term1 = np.transpose(term1a, (1,2,0,3))

term2a = term1a #np.tensordot(HL_avg_top, HL_delta, axes=((1),(2)))
term2b = np.tensordot(np.tensordot(term2a, HL_delta, axes=((2,3),(0,2))), HL_avg_top, axes=((3),(0)))
term2 = np.transpose(term2b, (1,2,0,3))

eta_kappa = term0 - term1 + term2; #(∞.71)

term = -1*np.tensordot(eta_kappa, (zz - yy), axes=((1,3),(0,1)))
delta_weight_00 = np.tensordot(term, input_dweight, axes=((0,1),(0,1))) * lw_input #(∞.68)
delta_bias_00 = np.tensordot(term, input_dbias, axes=((0,1),(0,1))) * lb #(∞.68)
delta_weight_01 = np.tensordot(term, hidden_dweight, axes=((0,1),(0,1))) * lw_hidden #(∞.68)
delta_bias_01 = np.tensordot(term, hidden_dbias, axes=((0,1),(0,1))) * lb #(∞.68)
delta_weight_02 = np.tensordot(term, output_dweight, axes=((0,1),(0,1))) * lw_output #(∞.68)
delta_bias_02 = np.tensordot(term, output_dbias, axes=((0,1),(0,1))) * lb #(∞.68)

prediction_beforesteps = logits.argmax(dim=-1)
accuracy_beforesteps = calculate_accuracy(prediction_beforesteps, labels)
print("Before steps: accuracy={}, MSE={}".format(accuracy_beforesteps, mean_squared_error(yy, zz)))


Before steps: accuracy=0.03125, MSE=9.806567485580384


In [42]:
#input_fc_weight, input_fc_bias, hidden_fc_weight, hidden_fc_bias, output_fc_weight, output_fc_bias = \
#    testNet.input_fc.weight, testNet.input_fc.bias, testNet.hidden_fc.weight, testNet.hidden_fc.bias\
#        , testNet.output_fc.weight, testNet.output_fc.bias

#Step forward
coeff = 0.4
with torch.no_grad():
    testNet.input_fc.weight += torch.from_numpy(delta_weight_00) * coeff
    testNet.input_fc.bias += torch.from_numpy(delta_bias_00) * coeff
    testNet.hidden_fc.weight += torch.from_numpy(delta_weight_01) * coeff
    testNet.hidden_fc.bias += torch.from_numpy(delta_bias_01) * coeff
    testNet.output_fc.weight += torch.from_numpy(delta_weight_02) * coeff
    testNet.output_fc.bias += torch.from_numpy(delta_bias_02) * coeff
    logits_step0 = testNet.forward_(xx)
    prediction_step0 = logits_step0.argmax(dim=-1)

#Checking accuracy on trainset
accuracy_step0 = calculate_accuracy(prediction_step0, labels)
print("After step0: accuracy={}, MSE={}"\
      .format(accuracy_step0, mean_squared_error(yy, np.transpose(logits_step0.detach().numpy()))))

After step0: accuracy=0.234375, MSE=7.710322990236796


In [43]:
#After step0: accuracy=0.84375, MSE=3.9746734145347524
#After step1: accuracy=0.96875, MSE=1.559123651305658
test_accuracy_meter = AverageMeter()

for test_images, test_labels in test_dataloader:
    test_batch_size = test_images.shape[0]
    test_xx = test_images.view(test_batch_size, -1)
    with torch.no_grad():
        test_logits = testNet.forward_(test_xx)
        test_prediction = test_logits.argmax(dim=-1)

    test_accuracy_meter.update(calculate_accuracy(test_prediction, test_labels))

print("Accuracy on test-set:{}".format(test_accuracy_meter.avg))

Accuracy on test-set:0.13853503184713375


##### NTK re-calc

In [44]:
testNet.grad_zero
logits0 = testNet.forward_(xx)
zz0 = np.transpose(logits0.detach().numpy())

print("Calculating derivatives")
input_dweight0 = np.zeros((OUTPUT_DIM, BATCH_SIZE, 250, INPUT_DIM))
input_dbias0 = np.zeros((OUTPUT_DIM, BATCH_SIZE, 250))
hidden_dweight0 = np.zeros((OUTPUT_DIM, BATCH_SIZE, 100, 250))
hidden_dbias0 = np.zeros((OUTPUT_DIM, BATCH_SIZE, 100))
output_dweight0 = np.zeros((OUTPUT_DIM, BATCH_SIZE, OUTPUT_DIM, 100))
output_dbias0 = np.zeros((OUTPUT_DIM, BATCH_SIZE, OUTPUT_DIM))

for kk, alpha in itr.product(range(OUTPUT_DIM), range(BATCH_SIZE)):
    df = torch.autograd.grad(logits0[alpha,kk], (testNet.input_fc.weight, testNet.input_fc.bias\
                                    , testNet.hidden_fc.weight, testNet.hidden_fc.bias\
                                    , testNet.output_fc.weight, testNet.output_fc.bias)\
        , retain_graph=True, create_graph=True, allow_unused=True)
    input_dweight0[kk, alpha] = df[0].detach().numpy()
    input_dbias0[kk, alpha] = df[1].detach().numpy()
    hidden_dweight0[kk, alpha] = df[2].detach().numpy()
    hidden_dbias0[kk, alpha] = df[3].detach().numpy()
    output_dweight0[kk, alpha] = df[4].detach().numpy()
    output_dbias0[kk, alpha] = df[5].detach().numpy()

print("Calculating NTK by (8.4)")
HL0 = np.zeros((OUTPUT_DIM, OUTPUT_DIM, BATCH_SIZE, BATCH_SIZE))
for kk1, kk2, alpha1, alpha2 in itr.product(range(OUTPUT_DIM), range(OUTPUT_DIM)\
                                            , range(BATCH_SIZE), range(BATCH_SIZE)):
    if HL0[kk1, kk2, alpha1, alpha2] == 0 or HL0[kk2, kk1, alpha2, alpha1] == 0:    
        val = lb * np.dot(input_dbias0[kk1, alpha1], input_dbias0[kk2, alpha2])
        val += lw_input * np.sum(np.multiply(input_dweight0[kk1, alpha1], input_dweight0[kk2, alpha2]))
        val += lb * np.dot(hidden_dbias0[kk1, alpha1], hidden_dbias0[kk2, alpha2])
        val += lw_hidden * np.sum(np.multiply(hidden_dweight0[kk1, alpha1], hidden_dweight0[kk2, alpha2]))
        val += lb * np.dot(output_dbias0[kk1, alpha1], output_dbias0[kk2, alpha2])
        val += lw_output * np.sum(np.multiply(output_dweight0[kk1, alpha1], output_dweight0[kk2, alpha2]))
        HL0[kk1, kk2, alpha1, alpha2] = HL0[kk2, kk1, alpha2, alpha1] = val

print("Calculating average and inverted")
HL0_avg = np.zeros((BATCH_SIZE, BATCH_SIZE))
#HL_delta = np.copy(HL)

for alpha1, alpha2 in itr.product(range(BATCH_SIZE), range(BATCH_SIZE)):
    H0_avg = np.average([HL[num, num, alpha1, alpha2] for num in np.arange(OUTPUT_DIM)])
    HL0_avg[alpha1, alpha2] = H0_avg
    #for num in np.arange(OUTPUT_DIM):
    #    HL_delta[num, num, alpha1, alpha2] = HL[num, num, alpha1, alpha2] - H_avg

HL0_avg_top = np.linalg.inv(HL0_avg)


Calculating derivatives
Calculating NTK by (8.4)
Calculating average and inverted


##### Small step calculation

In [45]:
#lw_input, lw_hidden, lw_output
term0 = -1*np.matmul((zz0 - yy), HL0_avg_top)
delta_weight_10 = np.tensordot(term0, input_dweight0, axes=([0,1],[0,1])) * lw_input
delta_bias_10 = np.tensordot(term0, input_dbias0, axes=([0,1],[0,1])) * lb
delta_weight_11 = np.tensordot(term0, hidden_dweight0, axes=([0,1],[0,1])) * lw_hidden
delta_bias_11 = np.tensordot(term0, hidden_dbias0, axes=([0,1],[0,1])) * lb
delta_weight_12 = np.tensordot(term0, output_dweight0, axes=([0,1],[0,1])) * lw_output
delta_bias_12 = np.tensordot(term0, output_dbias0, axes=([0,1],[0,1])) * lb


In [46]:
#Step forward
coeff1 = 0.2
with torch.no_grad():
    testNet.input_fc.weight += torch.from_numpy(delta_weight_10) * coeff1
    testNet.input_fc.bias += torch.from_numpy(delta_bias_10) * coeff1
    testNet.hidden_fc.weight += torch.from_numpy(delta_weight_11) * coeff1
    testNet.hidden_fc.bias += torch.from_numpy(delta_bias_11) * coeff1
    testNet.output_fc.weight += torch.from_numpy(delta_weight_12) * coeff1
    testNet.output_fc.bias += torch.from_numpy(delta_bias_12) * coeff1
    logits_step1 = testNet.forward_(xx)
    prediction_step1 = logits_step1.argmax(dim=-1)

#Checking accuracy on trainset
accuracy_step1 = calculate_accuracy(prediction_step1, labels)
#print("Accuracy after step1:{}".format(accuracy_step1))
print("After step1: accuracy={}, MSE={}"\
      .format(accuracy_step1, mean_squared_error(yy, np.transpose(logits_step1.detach().numpy()))))

After step1: accuracy=0.5, MSE=3.734874306699859


In [47]:
test_accuracy_meter1 = AverageMeter()
for test_images, test_labels in test_dataloader:
    test_batch_size = test_images.shape[0]
    test_xx = test_images.view(test_batch_size, -1)
    with torch.no_grad():
        test_logits = testNet.forward_(test_xx)
        test_prediction = test_logits.argmax(dim=-1)

    test_accuracy_meter1.update(calculate_accuracy(test_prediction, test_labels))

print("Accuracy on test-set:{}".format(test_accuracy_meter1.avg))

Accuracy on test-set:0.20720541401273884


In [48]:
#Accuracy on test-set:0.3413538338658147
#Accuracy on test-set:0.38777955271565495
logits_step1

tensor([[-3.8656, -0.4131, -2.3410, -5.4354, -3.4970, -5.0222, -3.2681, -2.5629,
         -2.2123, -2.6724],
        [-3.4277, -0.9066, -2.6554, -5.1986, -3.8517, -5.2773, -2.0680, -4.4425,
         -1.8201, -0.7024],
        [-3.4497, -0.8314, -3.2819, -5.4710, -3.3301, -3.7837, -3.0626, -1.9794,
         -1.7329, -3.7457],
        [-1.3398, -0.7999, -2.3362, -5.5073, -4.9147, -6.4728, -2.5858, -6.2243,
         -1.9696, -2.2438],
        [-3.7139,  0.0224, -2.1681, -4.8242, -5.6869, -5.9020, -1.2467, -5.8000,
          0.9435, -3.8102],
        [-4.0503, -0.6230, -2.1414, -5.2521, -3.2004, -5.0207, -2.9355, -2.7622,
         -1.6966, -2.8429],
        [-3.5815, -0.1586, -3.4146, -5.4228, -4.4128, -5.7765, -2.1526, -5.3768,
         -2.1316, -0.5485],
        [-1.7775, -0.3249, -1.8834, -4.3274, -5.2525, -5.3838, -2.9755, -5.4472,
         -2.1826, -2.6519],
        [-2.4881, -0.4280, -1.9722, -3.4437, -0.3132, -3.2159, -3.3272, -2.1286,
         -1.2936, -2.0128],
        [-4.2442, -